In [ ]:
# Импорт необходимых модулей Flask и SQLAlchemy
from flask import Flask, render_template, request, redirect, url_for, flash  # Для работы с веб-приложением и отображения сообщений
from flask_sqlalchemy import SQLAlchemy  # Для работы с базой данных через ORM

# Инициализация приложения Flask
app = Flask(__name__)
app.secret_key = 'Oni_ybily_Kenny'  # Устанавливаем секретный ключ для шифрования flash-сообщений
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///users.db'  # Путь к базе данных SQLite
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False  # Отключение системы отслеживания изменений для повышения производительности

# Инициализация базы данных
db = SQLAlchemy(app)

# Определение модели пользователя, представляющей таблицу в базе данных
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)  # Уникальный идентификатор пользователя (первичный ключ)
    username = db.Column(db.String(80), nullable=False, unique=True)  # Имя пользователя, уникальное и обязательное
    email = db.Column(db.String(120), nullable=False, unique=True)  # Email пользователя, уникальный и обязательный
    password = db.Column(db.String(200), nullable=False)  # Пароль пользователя, обязательное поле

# Определение маршрута для главной страницы
@app.route('/')
def button_page():
    return render_template('button.html')  # Рендерит HTML-шаблон главной страницы

# Определение маршрута для страницы регистрации
@app.route('/register', methods=['GET', 'POST'])
def registration_page():
    if request.method == 'POST':  # Если метод запроса POST, обрабатываем данные формы
        username = request.form['username']  # Получаем имя пользователя из формы
        email = request.form['email']  # Получаем email из формы
        password = request.form['password']  # Получаем пароль из формы

        # Проверяем, существует ли пользователь с таким именем или email
        existing_user = User.query.filter((User.username == username) | (User.email == email)).first()
        if existing_user:  # Если пользователь найден, выводим сообщение об ошибке
            flash('Имя пользователя или email уже заняты', 'error')
            return redirect(url_for('registration_page'))  # Перенаправляем обратно на страницу регистрации

        # Если пользователя нет, добавляем нового в базу данных
        new_user = User(username=username, email=email, password=password)
        db.session.add(new_user)  # Добавляем запись в сессию
        db.session.commit()  # Фиксируем изменения в базе данных

        flash('Регистрация прошла успешно!', 'success')  # Выводим сообщение об успешной регистрации
        return redirect(url_for('login_page'))  # Перенаправляем на страницу входа

    return render_template('index_registration.html')  # Если метод запроса GET, отображаем форму регистрации

# Определение маршрута для страницы входа
@app.route('/login', methods=['GET', 'POST'])
def login_page():
    if request.method == 'POST':  # Если метод запроса POST, обрабатываем данные формы
        email = request.form['email']  # Получаем email из формы
        password = request.form['password']  # Получаем пароль из формы

        # Проверяем, существует ли пользователь с введенными email и паролем
        user = User.query.filter_by(email=email, password=password).first()
        if user:  # Если пользователь найден, выводим приветствие
            flash(f'Добро пожаловать, {user.username}!', 'success')
            return redirect(url_for('button_page'))  # Перенаправляем на главную страницу
        else:  # Если пользователь не найден, выводим сообщение об ошибке
            flash('Неверный email или пароль', 'error')

    return render_template('index_login.html')  # Если метод запроса GET, отображаем форму входа

# Точка входа в приложение
if __name__ == '__main__':
    with app.app_context():  # Создаем контекст приложения
        db.create_all()  # Создаем таблицы в базе данных, если они еще не существуют
    app.run(debug=True)  # Запускаем приложение в режиме отладки
